# Tiny RNN for **Sequence Labeling** in PyTorch
 
**Task:** Build and train a *very small* RNN for **token-level (per‑timestep)** prediction — the core idea behind POS tagging, NER, etc.

**Goals:**
- Understand how a simple RNN cell computes a new hidden state from the current input and previous hidden state.
- Build a per-timestep sequence model for **sequence labeling** (one label per token).
- Train with `BCEWithLogitsLoss` and measure **per-token accuracy**.
- Read tensor **shapes** confidently: `(batch, seq_len, features)` in → `(batch, seq_len)` out.

**Learning roadmap:**
1. Tensors & autograd basics.
2. Why **mini‑batches** matter.
3. RNN cell math and shapes.
4. Implement an RNN **cell** (one step).
5. Implement a tiny **sequence** model that outputs one label per position.
6. Train on a simple per‑timestep labeling task (running parity).
7. Record your final per‑token accuracy.


---
## 0) Setup

If `import torch` fails, uncomment the `pip install` line. We set a random seed and detect GPU if available.


In [32]:
# If PyTorch is missing in your environment, uncomment:
# !pip -q install torch

import random
from typing import Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)


PyTorch: 2.10.0+cu128
Device: cpu


## 1) Mini crash course: Tensors, Autograd, and a training step

**Tensors** are multi‑dimensional arrays. With `requires_grad=True`, PyTorch tracks operations so it can compute **gradients** during `loss.backward()`.

**Tensor:** like a NumPy array, but can track gradients.
**`requires_grad=True`:** PyTorch keeps a computation graph so that calling `loss.backward()` computes gradients for you.
> **Training step (in words):**
> 1) Forward: `x` → `y_pred` → `loss`
> 2) Backward: `loss.backward()` populates `param.grad`
> 3) Update: `optimizer.step()` nudges parameters to reduce loss
>
> **Sanity check:** In the next code cell we show this on a 1-layer linear model with mean-squared error.


In [33]:
# Tiny autograd demo
x = torch.tensor([[1.0, 2.0, 3.0]], requires_grad=True)  # (1, 3)
W = torch.randn(3, 2) * 0.1                              # (3, 2) parameters
b = torch.zeros(2)                                       # (2,) parameters

y = x @ W + b            # -> (1, 2)
loss = (y**2).mean()     # scalar
print("y:", y)
print("loss:", loss)

loss.backward()          # compute d(loss)/d(...)
print("x.grad:", x.grad.shape)
print("W.grad:", W.grad.shape if W.grad is not None else None) # None because we didn't specify it with requires_grad=True
print("b.grad:", b.grad.shape if b.grad is not None else None) # None because we didn't specify it with requires_grad=True


y: tensor([[ 0.6819, -0.4127]], grad_fn=<AddBackward0>)
loss: tensor(0.3177, grad_fn=<MeanBackward0>)
x.grad: torch.Size([1, 3])
W.grad: None
b.grad: None


## 2) Why **batches**?

Most datasets are large. We almost never update parameters with a **single example** (too noisy/slow) or the **entire dataset** (too slow / too much memory). We use **mini‑batches** (e.g., 32 or 64 examples).

**Benefits:**
1. **Speed via parallelism:** one large batched matrix multiply is faster than many tiny ones.
2. **Stabler gradients:** batches average noise across examples.
3. **Memory control:** batches fit into RAM/GPU memory while the full dataset might not.

**Our shapes with batches:**
- Input `x`: **(batch, seq_len, input_size)**  
- Hidden state `h`: **(batch, hidden_size)**  
- Per‑step logits `o`: **(batch, seq_len)**  
- Per‑step labels `y`: **(batch, seq_len)**

> In real NLP we handle variable lengths via padding + masks. Here all sequences are same length for clarity.


In [34]:
# Vectorization demo: batch vs loop
B, D_in, D_out = 4, 5, 3
Xb = torch.randn(B, D_in)
W = torch.randn(D_in, D_out) * 0.1
b = torch.zeros(D_out)

Y_vec = Xb @ W + b  # vectorized

Y_loop = []
for i in range(B):
    Y_loop.append(Xb[i:i+1] @ W + b)
Y_loop = torch.cat(Y_loop, dim=0)

print("Same?", torch.allclose(Y_vec, Y_loop, atol=1e-6))
print("Vectorized:", Y_vec.shape, "| Loop:", Y_loop.shape)


Same? True
Vectorized: torch.Size([4, 3]) | Loop: torch.Size([4, 3])


## 3) RNN concept & shapes (for sequence labeling)

An RNN processes a sequence one step at a time, carrying a **hidden state** \(h_t\) that summarizes what it has seen.

**RNN cell (vanilla tanh):**
\[
h_t = \tanh(x_t W_{xh} + h_{t-1} W_{hh} + b_h)
\]

For **sequence labeling**, we want one score (logit) **at every step**:
\[
o_t = h_t W_{hy} + b_y
\]

- `x_t` is the current token’s features (here, a single scalar 0/1 → `input_size=1`).
- The same weights are shared across all timesteps.
- In real NLP, `x_t` would be a word/char embedding vector.


### Shape legend

- `x` : **(batch, seq_len, input_size)**  
- `x[:, t, :]` : **(batch, input_size)** — batch of tokens at timestep `t`  
- `h` : **(batch, hidden_size)** — running memory  
- per‑step logits `o_t` : **(batch, 1)** (we squeeze to **(batch,)**, then stack across time)  
- labels `y` : **(batch, seq_len)**

We’ll use `BCEWithLogitsLoss` (sigmoid + binary cross‑entropy in one stable function).


## 4) Implement a tiny RNN **cell** (one step)

**Your task (TODO 1)**: compute `h_t` from `x_t` and `h_prev` using the formula above.  
Then run the **solution** cell to proceed.


In [35]:
# === TODO 1 (student): TinyRNNCell ===
class TinyRNNCell(nn.Module):
    """
    One RNN step:
      h_t = tanh(x_t @ W_xh + h_prev @ W_hh + b_h)
    """
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W_xh = nn.Parameter(torch.randn(input_size, hidden_size) * 0.1)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)
        self.b_h  = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        # Hints:
        # - use torch.tanh(...)
        # - x_t @ W_xh -> (batch, hidden), h_prev @ W_hh -> (batch, hidden)
        # return torch.zeros_like(h_prev)  # placeholder if you want to run
        
        h_t = torch.tanh(x_t @ self.W_xh + h_prev @ self.W_hh + self.b_h)
        return h_t

### Quick test for the cell

In [36]:
torch.manual_seed(0)
B, I, H = 3, 1, 4
cell = TinyRNNCell(I, H)
with torch.no_grad():
    cell.W_xh.fill_(1.0)  # ignore h_prev
    cell.W_hh.zero_()
    cell.b_h.zero_()
x_t = torch.tensor([[0.0],[1.0],[2.0]])
h_prev = torch.randn(B, H)
expected = torch.tanh(x_t @ torch.ones(I, H))
out = cell(x_t, h_prev)
print("max |diff|:", float((out - expected).abs().max()))
assert torch.allclose(out, expected, atol=1e-6)
print("✅ TinyRNNCell test passed.")


max |diff|: 0.0
✅ TinyRNNCell test passed.


## 5) Tiny RNN **Sequence** model (per‑timestep outputs) — with extra guidance

We now **unroll** the cell over the sequence and emit a logit at **each** timestep.

### What is `W_hy`?
`W_hy` is a small weight matrix of shape **(hidden_size, 1)** that **projects the hidden state** \(h_t\) at each timestep into a **single scalar** (the logit). Think of it as a tiny linear classifier that reads the memory `h_t` and says **"is the label 1 at this position?"**. We **reuse** the same `W_hy` at every timestep (weights are shared through time).

### How to write the for‑loop
1. Start with `h = zeros(batch, hidden_size)` on the same `device` as `x`.
2. For each timestep `t = 0 .. seq_len-1`:
   - Extract the batch of inputs at that step: `x_t = x[:, t, :]` (shape `(batch, input_size)`).
   - Update the hidden state: `h = cell(x_t, h)`.
   - Compute the per‑step logit: `o_t = h @ W_hy + b_y` → shape `(batch, 1)`.
   - `squeeze(-1)` to make `(batch,)` and **append** to a Python list.
3. After the loop, `torch.stack(collected, dim=1)` to get `(batch, seq_len)`.


In [38]:
# === TODO 2 (student): TinyRNNSequence ===
class TinyRNNSequence(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.cell = TinyRNNCell(input_size, hidden_size)
        self.hidden_size = hidden_size
        self.W_hy = nn.Parameter(torch.randn(hidden_size, 1) * 0.1)  # projection H->1
        self.b_y  = nn.Parameter(torch.zeros(1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, seq_len, input_size)
        returns logits: (batch, seq_len)
        """
        B, T, _ = x.shape
        h = torch.zeros(B, self.hidden_size, device=x.device)
        logits_per_t = []
        """More code here"""
        
        for t in range(T):
            x_t = x[:, t, :]
            h = self.cell(x_t, h)
            o_t = h @ self.W_hy + self.b_y
            logits_per_t.append(o_t.squeeze(-1))
        
        return torch.stack(logits_per_t, dim=1)


### Quick test for the sequence model

In [39]:
torch.manual_seed(0)
B, T, I, H = 4, 6, 1, 1
x = torch.randn(B, T, I)
model = TinyRNNSequence(I, H)
with torch.no_grad():
    model.cell.W_xh.fill_(1.0)
    model.cell.W_hh.zero_()
    model.cell.b_h.zero_()
    model.W_hy.fill_(1.0)
    model.b_y.zero_()
logits = model(x)                     # (B, T)
expected = torch.tanh(x.squeeze(-1))  # (B, T)
print("max |diff|:", float((logits - expected).abs().max()))
assert torch.allclose(logits, expected, atol=1e-6)
print("✅ TinyRNNSequence test passed.")


max |diff|: 0.0
✅ TinyRNNSequence test passed.


## 6) Dataset: **Running parity** with per‑position labels

**Inputs:** sequences of 0/1 bits `(batch, seq_len, 1)`  
**Targets:** at position *t*, label is 1 if the number of 1s up to *t* is odd, else 0 → `(batch, seq_len)`

**Tiny example (running parity):**  
Input bits `X = [1, 0, 1, 1]` → running sum = `[1, 1, 2, 3]` → parity (odd=1, even=0) = **`[1, 1, 0, 1]`**.  
That is the target **per position**.

This mimics token‑level NLP tasks (per‑token labels).


In [40]:
class ParityPerStepDataset(Dataset):
    def __init__(self, n_samples: int = 8000, seq_len: int = 16):
        super().__init__()
        self.seq_len = seq_len
        X = torch.randint(0, 2, (n_samples, seq_len, 1)).float()     # 0/1 inputs
        Y = (X.squeeze(-1).cumsum(dim=1) % 2).float()                 # running parity
        self.X, self.Y = X, Y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# Loaders
train_ds = ParityPerStepDataset(n_samples=8000, seq_len=16)
test_ds  = ParityPerStepDataset(n_samples=2000, seq_len=16)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False)

print(f"Train: {len(train_ds)} | Test: {len(test_ds)}")


Train: 8000 | Test: 2000


In [41]:
# Peek at one tiny batch
peek_loader = DataLoader(ParityPerStepDataset(n_samples=2, seq_len=8), batch_size=2)
X_peek, Y_peek = next(iter(peek_loader))
print("X (bits):\n", X_peek.squeeze(-1).int())
print("Y (running parity):\n", Y_peek.int())
assert X_peek.shape == (2, 8, 1) and Y_peek.shape == (2, 8)


X (bits):
 tensor([[0, 0, 1, 1, 0, 0, 1, 0],
        [0, 0, 0, 0, 1, 1, 0, 1]], dtype=torch.int32)
Y (running parity):
 tensor([[0, 0, 1, 0, 0, 0, 1, 1],
        [0, 0, 0, 0, 1, 0, 0, 1]], dtype=torch.int32)


## 7) Training for sequence labeling (one unified skeleton)

We define:
- `token_accuracy` to compute per‑token accuracy.
- A single **training skeleton** `train(...)` that takes `optimizer` and `criterion` as **arguments**. `criterion` is the `BCEWithLogitsLoss` loss function that we saw earlier.

> **Per-token accuracy:** After computing logits `(B, T)`, we predict `1` if `sigmoid(logit) >= 0.5`, else `0`, and take the fraction equal to the gold labels.
>
> **Expected curve:** Training loss should go **down**; accuracy should go **up** to ≥95% within ~10 epochs on CPU for this toy task.

In [42]:
def token_accuracy(model: nn.Module, data_loader: DataLoader, device=device) -> float:
    """Compute per-token accuracy with sigmoid+0.5 thresholding.

    Returns:
        float in [0,1]: (# correct tokens) / (total tokens)
    """
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, Y in data_loader:
            X, Y = X.to(device), Y.to(device)
            logits = model(X)                        # (B, T)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == Y).sum().item()
            total += Y.numel()
    return correct / total


In [43]:
# === TODO 3–5 (student): unified training skeleton ===
def train_skeleton(model: nn.Module,
                   train_loader: DataLoader,
                   test_loader: DataLoader,
                   optimizer: optim.Optimizer,
                   criterion: nn.Module,
                   epochs: int = 8,
                   device=device):
    model.train()
    for epoch in range(1, epochs + 1):
        running_loss = 0.0
        for X, Y in train_loader:
            X, Y = X.to(device), Y.to(device)

            # TODO 3: forward pass to get logits (B, T)
            logits = model(X)

            # TODO 4: compute loss over all tokens
            loss = criterion(logits.view(-1), Y.view(-1).float())

            # TODO 5: standard optimizer step
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += float(loss.item()) * X.size(0)

        avg_loss = running_loss / len(train_loader.dataset)
        acc = token_accuracy(model, test_loader, device=device)
        print(f"[SKELETON] Epoch {epoch:02d} | loss: {avg_loss:.4f} | per‑token acc: {acc*100:.1f}%")


## 8) Built‑in tests

These checks validate shapes and confirm your unified training loop can overfit a tiny batch by calling it with `epochs=1` repeatedly.


In [44]:
# Test A: shapes
Xb, Yb = next(iter(train_loader))
assert Xb.dim() == 3 and Yb.dim() == 2 and Xb.shape[-1] == 1
print("✅ Shapes OK:", Xb.shape, Yb.shape)

# Test B: forward output shape
tmp_model = TinyRNNSequence(input_size=1, hidden_size=8)
out = tmp_model(Xb)
assert out.shape == Yb.shape, "Model output must be (batch, seq_len)"
print("✅ Model forward returns:", out.shape)


✅ Shapes OK: torch.Size([64, 16, 1]) torch.Size([64, 16])
✅ Model forward returns: torch.Size([64, 16])


In [45]:
# Test C: tiny overfit using the unified train() with epochs=1
tiny_ds = ParityPerStepDataset(n_samples=128, seq_len=10)
tiny_loader = DataLoader(tiny_ds, batch_size=128, shuffle=True)
small_model = TinyRNNSequence(input_size=1, hidden_size=8).to(device)
small_opt = optim.Adam(small_model.parameters(), lr=0.02)
crit = nn.BCEWithLogitsLoss()

for i in range(3):
    train_skeleton(small_model, tiny_loader, tiny_loader, small_opt, crit, epochs=1, device=device)
    acc = token_accuracy(small_model, tiny_loader)
    print(f"tiny overfit | after {i+1} epoch(s) | acc {acc*100:.1f}%")


[SKELETON] Epoch 01 | loss: 0.6934 | per‑token acc: 51.2%
tiny overfit | after 1 epoch(s) | acc 51.2%
[SKELETON] Epoch 01 | loss: 0.6927 | per‑token acc: 51.2%
tiny overfit | after 2 epoch(s) | acc 51.2%
[SKELETON] Epoch 01 | loss: 0.6923 | per‑token acc: 51.2%
tiny overfit | after 3 epoch(s) | acc 51.2%


## 9) **Final Exercise** — Train end-to-end, tune minimally, and report

**Goal:** Train a fresh model on the running-parity dataset and reach **≥ 90% per-token accuracy** on the **test** set.

### What to do
1. Instantiate a fresh model, optimizer, and loss.
2. Train for several epochs using the **solution** `train(...)` function.
3. Compute per‑token accuracy on the test set.
4. **Write down your final accuracy** and, in one sentence, mention any hyperparameter changes you tried.

### Suggested hyperparameters (start here)
- `hidden_size`: **16** (try **32** if underperforming)
- `learning_rate`: **0.01** (try **0.005** if loss is noisy; **0.02** if learning is too slow)
- `epochs`: **10** (increase to **15–20** if accuracy plateaus early)
- `batch_size`: keep default (only try **32** or **128** if you must)

> **Quick tuning checklist (only if <=90%)**
> - Lower LR 2–3× if loss is bouncing; raise LR 2× if loss barely moves.
> - Increase `hidden_size` from 16 → 32 for more capacity.
> - Add a few more epochs (e.g., +5).
> - Re-check that you **didn’t** apply `sigmoid` inside the model (we use `BCEWithLogitsLoss`).

In [46]:
# Fresh model/optimizer for the final run
hidden_size = 32
model = TinyRNNSequence(input_size=1, hidden_size=hidden_size).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.02)

print("Before training — test token acc:", token_accuracy(model, test_loader))
train_skeleton(model, train_loader, test_loader, optimizer, criterion, epochs=13, device=device)
final_acc = token_accuracy(model, test_loader)
print("After training — FINAL test token accuracy:", final_acc)

# === Record your result here ===
# Final test per-token accuracy: ________
#After training — FINAL test token accuracy: 1.0


Before training — test token acc: 0.5644375
[SKELETON] Epoch 01 | loss: 0.6906 | per‑token acc: 56.1%
[SKELETON] Epoch 02 | loss: 0.6314 | per‑token acc: 59.6%
[SKELETON] Epoch 03 | loss: 0.5771 | per‑token acc: 63.2%
[SKELETON] Epoch 04 | loss: 0.5211 | per‑token acc: 60.6%
[SKELETON] Epoch 05 | loss: 0.4638 | per‑token acc: 90.6%
[SKELETON] Epoch 06 | loss: 0.0495 | per‑token acc: 100.0%
[SKELETON] Epoch 07 | loss: 0.0124 | per‑token acc: 100.0%
[SKELETON] Epoch 08 | loss: 0.0073 | per‑token acc: 100.0%
[SKELETON] Epoch 09 | loss: 0.0050 | per‑token acc: 100.0%
[SKELETON] Epoch 10 | loss: 0.0037 | per‑token acc: 100.0%
[SKELETON] Epoch 11 | loss: 0.0029 | per‑token acc: 100.0%
[SKELETON] Epoch 12 | loss: 0.0024 | per‑token acc: 100.0%
[SKELETON] Epoch 13 | loss: 0.0019 | per‑token acc: 100.0%
After training — FINAL test token accuracy: 1.0


---
### You’re done!

- You implemented an RNN cell and a sequence model that emits **one label per token**.
- You trained it end‑to‑end and measured **per‑token accuracy**.
- You learned why **batches** matter for performance and stability.

Great job! 🎉


## Troubleshooting & FAQ

**Accuracy stuck ~50%**
- Don’t apply `sigmoid` inside the model; use `BCEWithLogitsLoss`.
- Check label dtype: labels must be float (0.0/1.0), same shape as logits.
- Try a slightly larger `hidden_size` (e.g., 32) or `lr=0.005 → 0.003`.


**Shapes don’t match errors**
- Inputs must be `(B, T, D_in)`; we index with `x[:, t, :]` to get `(B, D_in)`.
- Hidden state is `(B, H)`. Output logits are `(B, T)`.
- Revisit the shape table in Section 3.


**Loss not decreasing**
- Lower the learning rate by 2–3×.
- Verify the dataset generation with the parity example and print a few `(x, y)` pairs.